<a href="https://www.kaggle.com/code/sudeozmen/met-qa-final?scriptVersionId=366612369" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install -q "numpy<2"
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q langchain==0.2.14 langchain-community==0.2.12 langchain-core==0.2.35 langchain-text-splitters==0.2.2 langchain-huggingface==0.0.3 faiss-cpu rank_bm25 sentence-transformers psutil tiktoken langchain-cohere pydantic>=2.7.0

In [1]:
import os
import warnings
import torch

warnings.filterwarnings("ignore") 

from langchain_core.documents import Document  
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.retrievers import BM25Retriever


from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

print(f"GPU Aktif mi?: {torch.cuda.is_available()}")

GPU Aktif mi?: True


In [ ]:
DATA_PATH = "/kaggle/input/datasets/sudeozmen/the-met-knowledge-base/The_Met_Knowledge_Base"
documents = []
for filename in os.listdir(DATA_PATH):
    if filename.endswith(".txt"):
        file_path = os.path.join(DATA_PATH, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
            documents.append(Document(page_content=text, metadata={"source": filename}))

print(f"Toplam {len(documents)} adet doküman başarıyla yüklendi.")

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
import pandas as pd

EXCEL_PATH = "/kaggle/input/datasets/sudeozmen/met-rag-qa/MET_QA.xlsx"

documents = []
for filename in os.listdir(DATA_PATH):
    if filename.endswith(".txt"):
        with open(os.path.join(DATA_PATH, filename), 'r', encoding='utf-8') as f:
            documents.append(Document(page_content=f.read(), metadata={"source": filename}))

print(f" Toplam Doküman Sayısı: {len(documents)}")

df_sorular = pd.read_excel(EXCEL_PATH)

# ---------------------------------------------------------
# 2. CHUNKING VE EMBEDDING (BGE vs MiniLM / Recursive vs Fixed)
# ---------------------------------------------------------
chunker_1 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs_recursive = chunker_1.split_documents(documents)

chunker_2 = CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=0)
docs_fixed = chunker_2.split_documents(documents)

emb_bge = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", model_kwargs={'device': 'cuda'})
emb_mini = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})

# ---------------------------------------------------------
# 3. RETRIEVAL AYARLARI
# ---------------------------------------------------------
# Strateji 1: Standart Kosinüs Benzerliği
retrieval_strategy_1 = {"type": "similarity", "kwargs": {"k": 5}}

# Strateji 2: MMR (Maksimum Çeşitlilik - Aynı şeyleri tekrar etmesini önler)
retrieval_strategy_2 = {"type": "mmr", "kwargs": {"k": 5, "fetch_k": 20}}

# Strateji 3: Benzerlik Eşiği (Sadece %50'den fazla benzeyenleri getir, halüsinasyonu kes)
retrieval_strategy_3 = {"type": "similarity_score_threshold", "kwargs": {"score_threshold": 0.5, "k": 5}}

retrieval_stratejileri = [
    ("Standart Benzerlik", retrieval_strategy_1),
    ("MMR (Çeşitlilik)", retrieval_strategy_2),
    ("Score Threshold (Skor Eşiği)", retrieval_strategy_3)
]

# ---------------------------------------------------------
# 4. DEĞERLENDİRME FONKSİYONU
# ---------------------------------------------------------
def evaluate_config(docs, embedding_model, search_type, search_kwargs, df):
    vectorstore = FAISS.from_documents(docs, embedding_model)
    retriever = vectorstore.as_retriever(search_type=search_type, search_kwargs=search_kwargs)
    
    hit, mrr, toplam = 0, 0, 0
    
    for _, row in df.iterrows():
        if str(row['SORU TİPİ']).strip().upper() == "OOD": continue
        
        toplam += 1
        query = row['SORU']
        target = str(row['DOKUMAN NO']).strip()
        
        results = retriever.invoke(query)
        retrieved_names = [d.metadata.get('source', '').strip() for d in results]
        
        if target in retrieved_names:
            hit += 1
            rank = retrieved_names.index(target) + 1
            mrr += (1 / rank)
            
    hit_rate = (hit / toplam) * 100
    mrr_score = (mrr / toplam)
    return round(hit_rate, 2), round(mrr_score, 4)

# ---------------------------------------------------------
# 5.KOMBİNASYON DÖNGÜSÜ (2 Chunking x 2 Embedding x 3 Retrieval = 12 Kombinasyon)
# ---------------------------------------------------------
print("Kombinasyon Analizi Başlıyor...\n")

kombinasyonlar = [
    {"Ad": "BGE-Large + Recursive", "Docs": docs_recursive, "Emb": emb_bge},
    {"Ad": "BGE-Large + Fixed", "Docs": docs_fixed, "Emb": emb_bge},
    {"Ad": "MiniLM + Recursive", "Docs": docs_recursive, "Emb": emb_mini},
    {"Ad": "MiniLM + Fixed", "Docs": docs_fixed, "Emb": emb_mini}
]

sonuclar = []

for komb in kombinasyonlar:
    for strat_isim, strat_ayarlar in retrieval_stratejileri:
        print(f"Test ediliyor: {komb['Ad']} | Algoritma: {strat_isim}...")
        
        hit_r, mrr_s = evaluate_config(
            komb["Docs"], 
            komb["Emb"], 
            strat_ayarlar["type"], 
            strat_ayarlar["kwargs"], 
            df_sorular
        )
        
        sonuclar.append({
            "Embedding + Chunking": komb["Ad"],
            "Arama Algoritması": strat_isim,
            "Hit Rate (%)": hit_r,
            "Recall (%)": hit_r, 
            "MRR": mrr_s
        })


# ---------------------------------------------------------
# 6.FİNAL TABLOSU
# ---------------------------------------------------------
df_sonuclar = pd.DataFrame(sonuclar)
print("\n" + "="*85)
print("KOMBİNASYON TABLOSU")
print("="*85)
print(df_sonuclar.to_string(index=False))
print("="*85)

In [ ]:
print("\nLLM aşaması için global Retriever hafızaya alınıyor...")
# Tabloda genelde en iyi çıkan konfigürasyon (BGE + Recursive + Similarity) LLM için sabitleniyor:
global_vectorstore = FAISS.from_documents(docs_recursive, emb_bge)
retriever = global_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
print("Retriever hazır")

In [ ]:
import os
import time
import gc
import psutil 
import torch
import pandas as pd
import warnings
from kaggle_secrets import UserSecretsClient
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_community.llms import HuggingFacePipeline
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. ŞİFRELERİ ALMA 
# ---------------------------------------------------------
user_secrets = UserSecretsClient()
try: hf_token = user_secrets.get_secret("HF_TOKEN")
except: hf_token = None

try:
    cohere_api_key = user_secrets.get_secret("COHERE_API_KEY")
    os.environ["COHERE_API_KEY"] = cohere_api_key
except: cohere_api_key = None

# ---------------------------------------------------------
# 2. MODEL LİSTESİ (4 Lokal + 1 API)
# ---------------------------------------------------------
MODELS = [
    {"name": "Cohere-API", "id": "command-r-plus-08-2024", "type": "api"}, 
    {"name": "Llama-3-8B", "id": "meta-llama/Meta-Llama-3-8B-Instruct", "type": "local"}, 
    {"name": "Gemma-7B", "id": "google/gemma-1.1-7b-it", "type": "local"}, 
    {"name": "Mistral-7B", "id": "mistralai/Mistral-7B-Instruct-v0.2", "type": "local"}, 
    {"name": "Qwen-1.5-7B", "id": "Qwen/Qwen1.5-7B-Chat", "type": "local"} 
]

EXCEL_PATH = "/kaggle/input/datasets/sudeozmen/met-rag-qa/MET_QA.xlsx" 
OUTPUT_CSV = "Final_LLM_Karsilastirma_Tablosu.csv"

prompt_base = PromptTemplate(template="Answer the question accurately. If you don't know, strictly say 'I don't know'.\nQuestion: {question}\nAnswer:", input_variables=["question"])
prompt_rag = PromptTemplate(template="Use ONLY the following context to answer. If not in context, say 'I don't know'.\nContext:\n{context}\nQuestion: {question}\nAnswer:", input_variables=["context", "question"])

# ---------------------------------------------------------
# 3. YARDIMCI FONKSİYONLAR (VRAM, RAM, F1)
# ---------------------------------------------------------
def get_vram(): return torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
def get_sys_ram(): return psutil.virtual_memory().used / (1024**3) 

def calc_f1_em(pred, truth):
    pred_w = set(pred.lower().translate(str.maketrans('', '', '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')).split())
    truth_w = set(truth.lower().translate(str.maketrans('', '', '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')).split())
    em = 1.0 if truth.lower() in pred.lower() else 0.0
    if not pred_w or not truth_w: return em, 0.0
    common = pred_w.intersection(truth_w)
    precision = len(common) / len(pred_w) if len(pred_w) > 0 else 0
    recall = len(common) / len(truth_w) if len(truth_w) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return em, f1

# ---------------------------------------------------------
# 4. DEĞERLENDİRME DÖNGÜSÜ
# ---------------------------------------------------------
df_qa = pd.read_excel(EXCEL_PATH)
all_results = []

print("\n ANALİZ BAŞLIYOR...\n")

for model_info in MODELS:
    m_name = model_info["name"]
    m_type = model_info["type"]
    m_id = model_info["id"]
    
    print(f"\n{'='*60}\n {m_name} ({m_type.upper()}) YÜKLENİYOR...\n{'='*60}")
    
    if m_type == "local":
        tokenizer = AutoTokenizer.from_pretrained(m_id, token=hf_token, trust_remote_code=True)
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        
        # device_map={"": 0} yapılarak VRAM patlaması engellendi
        model = AutoModelForCausalLM.from_pretrained(
            m_id, 
            device_map={"": 0}, 
            quantization_config=bnb_config, 
            token=hf_token,
            trust_remote_code=True
        )
        
        pipe = pipeline(
            "text-generation", 
            model=model, 
            tokenizer=tokenizer, 
            max_new_tokens=150, 
            temperature=0.1, 
            do_sample=True, 
            return_full_text=False
        )
        base_llm = HuggingFacePipeline(pipeline=pipe)
        cost_per_query = 0.0
    else:
        base_llm = ChatCohere(model=m_id, temperature=0.1, cohere_api_key=cohere_api_key)
        cost_per_query = 0.001
        tokenizer = None

    chain_base = prompt_base | base_llm | StrOutputParser()
    chain_rag = prompt_rag | base_llm | StrOutputParser()
    
    vram_used = get_vram()
    sys_ram_used = get_sys_ram()
    
    for i, row in df_qa.iterrows():
        soru = str(row['SORU']).strip()
        

        gold_cevap = str(row.get('DOĞRU CEVAP', row.get('GOLD CEVAP', ''))).strip()
            
        soru_tipi = str(row.get('SORU TİPİ', 'NORMAL')).strip().upper() 
        
        # --- A) LLM-ONLY ---
        t0 = time.time()
        ans_base = chain_base.invoke({"question": soru}).strip()
        lat_base = time.time() - t0
        em_base, f1_base = calc_f1_em(ans_base, gold_cevap)
        
        if m_type == "local": tok_base = len(tokenizer.encode(ans_base)) / lat_base if lat_base > 0 else 0
        else: tok_base = (len(ans_base.split()) * 1.3) / lat_base if lat_base > 0 else 0
        
        # --- B) RAG + LLM ---
        docs = retriever.invoke(soru)
        context = "\n\n".join([d.page_content for d in docs])
        t1 = time.time()
        ans_rag = chain_rag.invoke({"context": context, "question": soru}).strip()
        lat_rag = time.time() - t1
        em_rag, f1_rag = calc_f1_em(ans_rag, gold_cevap)
        
        if m_type == "local": tok_rag = len(tokenizer.encode(ans_rag)) / lat_rag if lat_rag > 0 else 0
        else: tok_rag = (len(ans_rag.split()) * 1.3) / lat_rag if lat_rag > 0 else 0
        
        # --- HALÜSİNASYON ---
        hal_base = "Yok" if "i don't know" in ans_base.lower() else "Yüksek"
        hal_rag = "Yok" if "i don't know" in ans_rag.lower() else "Yüksek"
        
        if soru_tipi == "OOD":
            if hal_base == "Yüksek": em_base, f1_base = 0.0, 0.0
            if hal_rag == "Yüksek": em_rag, f1_rag = 0.0, 0.0
        
        all_results.append({
            "Model": m_name, "Soru Tipi": soru_tipi, "Sistem": "LLM-Only (Base)", 
            "EM": em_base, "F1": f1_base, "Latency (s)": round(lat_base,2), "Token/s": round(tok_base,2), 
            "Hallucination": hal_base, "VRAM (GB)": round(vram_used, 2), "Sys RAM (GB)": round(sys_ram_used, 2), "Cost ($)": cost_per_query,
            "Relevance (1-5)": "", "Faithfulness (1-5)": "", "Fluency (1-5)": ""
        })
        
        all_results.append({
            "Model": m_name, "Soru Tipi": soru_tipi, "Sistem": "RAG + Base LLM", 
            "EM": em_rag, "F1": f1_rag, "Latency (s)": round(lat_rag,2), "Token/s": round(tok_rag,2), 
            "Hallucination": hal_rag, "VRAM (GB)": round(vram_used, 2), "Sys RAM (GB)": round(sys_ram_used, 2), "Cost ($)": cost_per_query,
            "Relevance (1-5)": "", "Faithfulness (1-5)": "", "Fluency (1-5)": ""
        })
        
        if (i+1) % 10 == 0: pd.DataFrame(all_results).to_csv(OUTPUT_CSV, index=False)

        if m_type == "api":
            time.sleep(6.5)

    if m_type == "local":
        del model, tokenizer, pipe, base_llm, chain_base, chain_rag
        gc.collect()
        torch.cuda.empty_cache()

# ---------------------------------------------------------
# 5. FİNAL TABLOSU
# ---------------------------------------------------------
df_final = pd.DataFrame(all_results)
df_final.to_csv(OUTPUT_CSV, index=False)

pivot_table = df_final.groupby(['Sistem', 'Model', 'Soru Tipi']).agg(
    EM_Ort=('EM', 'mean'),
    F1_Ort=('F1', 'mean'),
    Latency_Sn=('Latency (s)', 'mean'),
    Token_Sn=('Token/s', 'mean'),
    VRAM_GB=('VRAM (GB)', 'max'),
    SysRAM_GB=('Sys RAM (GB)', 'max'),
    Cost_Top=('Cost ($)', 'sum')
).round(3)

print("\n" + "="*110)
print("KARŞILAŞTIRMA TABLOSU (SİSTEM + STRESS TEST ANALİZİ)")
print("="*110)
print(pivot_table.to_string())
print("="*110)

In [ ]:

!pip install -q -U PyYAML
!pip install -q "numpy<2"
!pip install -q langchain-cohere

import os
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate
from kaggle_secrets import UserSecretsClient

# 2. ŞİFRELER VE API
user_secrets = UserSecretsClient()
cohere_api_key = user_secrets.get_secret("COHERE_API_KEY")
os.environ["COHERE_API_KEY"] = cohere_api_key

#Sentetik veri üretimi için
llm = ChatCohere(model="command-r-plus-08-2024", temperature=0.7, cohere_api_key=cohere_api_key)

# 3. PROMPT: Sentetik Veri Çoğaltma
prompt_template = """
You are an expert data augmentation assistant. I will give you a Question and its Answer. 
Your task is to generate 3 alternative versions of this Question-Answer pair using different wording, phrasing, or perspective, but keeping the core factual information EXACTLY the same.

Original Question: {question}
Original Answer: {answer}

Generate 3 variations in the exact same language as the original. 
Format your output STRICTLY as follows, with no extra text:
Q1: [Alternative Question 1]
A1: [Alternative Answer 1]
Q2: [Alternative Question 2]
A2: [Alternative Answer 2]
Q3: [Alternative Question 3]
A3: [Alternative Answer 3]
"""
prompt = PromptTemplate(template=prompt_template, input_variables=["question", "answer"])
chain = prompt | llm

# 4. VERİYİ OKUMA VE ÇOĞALTMA
EXCEL_PATH = "/kaggle/input/datasets/sudeozmen/met-rag-qa/MET_QA.xlsx" 
df = pd.read_excel(EXCEL_PATH)

augmented_data = []

print("Sentetik Veri Üretimi Başlıyor (Hedef: 500+ QA Çifti)...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    orig_q = str(row['SORU']).strip()
    
    # DOĞRU CEVAP sütununu  al
    orig_a = str(row.get('DOĞRU CEVAP', row.get('GOLD CEVAP', ''))).strip()
    
    # Orijinal veriyi listeye ekle
    augmented_data.append({"instruction": orig_q, "output": orig_a})
    
    try:
        # LLM ile 3 yeni versiyon üret
        response = chain.invoke({"question": orig_q, "answer": orig_a})
        lines = response.content.strip().split('\n')
        
        # LLM çıktısını ayrıştır
        q1, a1, q2, a2, q3, a3 = "", "", "", "", "", ""
        for line in lines:
            if line.startswith("Q1:"): q1 = line.replace("Q1:", "").strip()
            elif line.startswith("A1:"): a1 = line.replace("A1:", "").strip()
            elif line.startswith("Q2:"): q2 = line.replace("Q2:", "").strip()
            elif line.startswith("A2:"): a2 = line.replace("A2:", "").strip()
            elif line.startswith("Q3:"): q3 = line.replace("Q3:", "").strip()
            elif line.startswith("A3:"): a3 = line.replace("A3:", "").strip()
        
        if q1 and a1: augmented_data.append({"instruction": q1, "output": a1})
        if q2 and a2: augmented_data.append({"instruction": q2, "output": a2})
        if q3 and a3: augmented_data.append({"instruction": q3, "output": a3})
        
    except Exception as e:
        pass 

# 5.Train ve Validation Olarak Bölme
df_augmented = pd.DataFrame(augmented_data)
df_augmented = df_augmented.dropna().drop_duplicates()

print(f"\n Üretilen Toplam Soru Sayısı: {len(df_augmented)} ")

# %90 Eğitim (Train), %10 Doğrulama (Validation)
train_df, val_df = train_test_split(df_augmented, test_size=0.1, random_state=42)

# 6. MISTRAL İÇİN İNSTRUCTION FORMATINA (JSONL) ÇEVİRME
def save_to_jsonl(dataframe, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for _, row in dataframe.iterrows():
            # Mistral'in anlayacağı prompt formatı: <s>[INST] Soru [/INST] Cevap </s>
            formatted_text = f"<s>[INST] {row['instruction']} [/INST] {row['output']} </s>"
            f.write(json.dumps({"text": formatted_text}, ensure_ascii=False) + "\n")

save_to_jsonl(train_df, "train_dataset.jsonl")
save_to_jsonl(val_df, "validation_dataset.jsonl")

print("Veriler 'train_dataset.jsonl' ve 'validation_dataset.jsonl' olarak başarıyla kaydedildi!")


In [ ]:
!pip uninstall -y bitsandbytes triton

!pip install -q "numpy<2" "triton<3.0.0" bitsandbytes==0.43.1 transformers==4.43.3 trl==0.9.6 peft==0.11.1 accelerate==0.33.0 datasets==2.20.0

In [ ]:
import gc
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

import os
import time
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from kaggle_secrets import UserSecretsClient

# =========================================================
# 1. ŞİFRE VE MODEL BİLGİLERİ
# =========================================================
user_secrets = UserSecretsClient()
try: hf_token = user_secrets.get_secret("HF_TOKEN")
except: hf_token = None

model_id = "mistralai/Mistral-7B-Instruct-v0.2"
output_dir = "./mistral_finetuned"

print("\nFİNE-TUNİNG SÜRECİ BAŞLIYOR...\n")

# =========================================================
# 2. VERİ SETİNİN YÜKLENMESİ VE FORMATLANMASI
# =========================================================
dataset = load_dataset("json", data_files={"train": "/kaggle/input/datasets/sudeozmen/fine-tune-dataset/train_dataset.jsonl", "validation": "/kaggle/input/datasets/sudeozmen/fine-tune-dataset/validation_dataset.jsonl"})
print(f"Veri Seti Yüklendi: {len(dataset['train'])} Eğitim, {len(dataset['validation'])} Doğrulama verisi.")

def formatting_prompts_func(example):
    return example['text']

# =========================================================
# 3.4-BİT (QLoRA) AYARLARI
# =========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"{model_id} GPU'ya 4-bit olarak indiriliyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token 
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}, 
    token=hf_token
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# =========================================================
# 4. PEFT / LoRA AYARLARI
# =========================================================
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() 

# =========================================================
# 5. EĞİTİM PARAMETRELERİ
# =========================================================
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,              
    per_device_train_batch_size=2,   
    gradient_accumulation_steps=4,   
    optim="paged_adamw_32bit",       
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,              
    weight_decay=0.001,
    fp16=False,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch" 
)

# =========================================================
# 6. SFT TRAINER VE EĞİTİMİ BAŞLATMA
# =========================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,
    formatting_func=formatting_prompts_func 
)

print("\nEĞİTİM BAŞLIYOR...\n")
start_time = time.time()

trainer.train()

end_time = time.time()
print(f"\nEĞİTİM TAMAMLANDI! Toplam Süre: {round((end_time - start_time)/60, 2)} dakika.")

# =========================================================
# 7. EĞİTİLMİŞ MODELİ KAYDETME
# =========================================================
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"model '{output_dir}' klasörüne kaydedildi!")

In [ ]:
import shutil
shutil.make_archive("Mistral_Model", 'zip', "./mistral_finetuned")
print("Zip işlemi tamamlandı")

In [ ]:
!pip install -q "numpy<2"
!pip install -q -U transformers accelerate peft bitsandbytes datasets sentence-transformers langchain-community langchain-cohere faiss-cpu

In [ ]:
import os
import time
import json
import torch
import pandas as pd
from collections import Counter
import string

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate
from kaggle_secrets import UserSecretsClient

# =========================================================
# 1. ŞİFRE VE APİ AYARLARI
# =========================================================
user_secrets = UserSecretsClient()
cohere_api_key = user_secrets.get_secret("COHERE_API_KEY")
os.environ["COHERE_API_KEY"] = cohere_api_key

# Human Evaluation için Cohere Hakem Modeli
llm_judge = ChatCohere(model="command-r-plus-08-2024", temperature=0.0, cohere_api_key=cohere_api_key)

# =========================================================
# 2. FAISS + TokenSplitter + E5-Base
# =========================================================
DATA_PATH = "/kaggle/input/datasets/sudeozmen/the-met-knowledge-base/The_Met_Knowledge_Base"
FAISS_PATH = "./faiss_met_index"

print("FAISS Veritabanı Kuruluyor...")
txt_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".txt")]
all_docs = []
for filename in txt_files:
    loader = TextLoader(os.path.join(DATA_PATH, filename), encoding='utf-8')
    doc = loader.load()[0]
    doc.metadata["doc_id"] = filename.replace(".txt", "")
    all_docs.append(doc)

text_splitter = TokenTextSplitter(chunk_size=256, chunk_overlap=32)
chunks = text_splitter.split_documents(all_docs)

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

vector_db = FAISS.from_documents(chunks, embeddings)
vector_db.save_local(FAISS_PATH)
print(f"FAISS Veritabanı Hazır! Toplam Chunk: {vector_db.index.ntotal}")

# =========================================================
# 3. METRİK HESAPLAMA FONKSİYONLARI
# =========================================================
def normalize_text(text):
    text = str(text).lower().translate(str.maketrans('', '', string.punctuation))
    return text.strip()

def calc_em_f1(prediction, ground_truth):
    pred_norm = normalize_text(prediction)
    truth_norm = normalize_text(ground_truth)
    
    em = 1.0 if (truth_norm and truth_norm in pred_norm) else 0.0
    
    pred_tokens = pred_norm.split()
    truth_tokens = truth_norm.split()
    if not pred_tokens or not truth_tokens: return em, 0.0
    
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0: return em, 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return em, f1

judge_prompt = PromptTemplate(
    template="""Sen katı bir akademik jürisin. Aşağıdaki verileri inceleyip metrikleri SADECE sayılarla (0 ile 5 arası) puanla.
Bağlam: {context}
Orijinal (Gold) Cevap: {gold}
Modelin Verdiği Cevap: {answer}

1. Relevance (İlgililik 0-5): Modelin cevabı soruya/gold cevaba ne kadar ilgili?
2. Faithfulness (Sadakat 0-5): Modelin cevabı bağlama sadık mı, yoksa bağlam dışı mı?
3. Fluency (Akıcılık 0-5): Modelin cevabı dilbilgisi olarak ne kadar akıcı?
4. Hallucination (Halüsinasyon 0 veya 1): Eğer cevapta bağlamda veya gold cevapta olmayan UYDURMA bir bilgi varsa 1, yoksa 0 yaz.

Çıktıyı SADECE şu formatta ver (Boşluk bırakarak yan yana 4 rakam):
[Relevance] [Faithfulness] [Fluency] [Hallucination]
Örnek Çıktı: 4 5 5 0""",
    input_variables=["context", "gold", "answer"]
)
judge_chain = judge_prompt | llm_judge

def get_human_metrics(context, gold, answer):
    try:
        if not context: context = "BAĞLAM YOK (NO RAG)"
        result = judge_chain.invoke({"context": context, "gold": gold, "answer": answer}).content.strip()
        scores = [int(s) for s in result.split() if s.isdigit()]
        if len(scores) >= 4:
            return scores[0], scores[1], scores[2], scores[3] 
    except:
        pass
    return 0, 0, 0, 0

print("tüm test fonksiyonları belleğe yüklendi!")

In [ ]:
# =========================================================
# 5. TEST PİPELİNE: 4 SİSTEMİ KARŞILAŞTIRMA
# =========================================================
train_data = pd.read_json("/kaggle/input/datasets/sudeozmen/fine-tune-dataset/train_dataset.jsonl", lines=True)
val_data = pd.read_json("/kaggle/input/datasets/sudeozmen/fine-tune-dataset/validation_dataset.jsonl", lines=True)
qa_df = pd.concat([train_data, val_data])

def extract_qa(text):
    try:
        parts = text.split("[/INST]")
        instruction = parts[0].replace("<s>[INST]", "").strip()
        output = parts[1].replace("</s>", "").strip()
        return instruction, output
    except:
        return "", ""

qa_df[['instruction', 'output']] = qa_df['text'].apply(lambda x: pd.Series(extract_qa(x)))
qa_df = qa_df.drop(columns=['text'])

print(f"Test için {len(qa_df)} örnek hazır!")


# =========================================================
# MODEL YÜKLEME (QLoRA / 4-bit)
# =========================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
finetuned_path = "/kaggle/input/datasets/sudeozmen/mistral-model"

# 1. 4-bit yapılandırma ayrı bir nesne olarak tanımlandı
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# 2. Model bu konfigürasyonla yüklendi
print("Model 4-bit olarak yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)


ft_model = PeftModel.from_pretrained(base_model, finetuned_path)

print("Modeller başarıyla yüklendi!")

In [ ]:
import os
import gc
import time
import psutil
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate
from kaggle_secrets import UserSecretsClient
import string
from collections import Counter
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import TokenTextSplitter

print(" NİHAİ SİSTEM KARŞILAŞTIRMASI BAŞLIYOR...")

# =====================================================================
# 1. BİLEŞENLERİ YÜKLEME 
# =====================================================================
EXCEL_PATH = "/kaggle/input/datasets/sudeozmen/test-q/TESTQ.xlsx" 
BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
FINETUNED_PATH = "/kaggle/input/datasets/sudeozmen/mistral-model" 
FAISS_PATH = "/kaggle/input/datasets/sudeozmen/faiss-db"

user_secrets = UserSecretsClient()
cohere_api_key = user_secrets.get_secret("COHERE_API_KEY")
hf_token = user_secrets.get_secret("HF_TOKEN")

# Hakem LLM (Human Eval için Cohere)
llm_judge = ChatCohere(model="command-r-plus-08-2024", temperature=0.0, cohere_api_key=cohere_api_key)

# Modelleri 4-bit QLoRA formatında yükleme (VRAM taşmasını önlemek için)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto", token=hf_token)
ft_model = PeftModel.from_pretrained(base_model, FINETUNED_PATH)

try:
    df_qa = pd.read_csv(EXCEL_PATH)
except:
    df_qa = pd.read_excel("/kaggle/input/datasets/sudeozmen/test-q/TESTQ.xlsx")

df_qa = df_qa.rename(columns={
    'Question': 'instruction', 
    'Gold Answer': 'output',
    'Q_Type': 'question_type',
    'Doc_No': 'doc_id'
})

print(f" {len(df_qa)} adet test sorusu yüklendi.")

print("Hazır FAISS Veritabanı Yükleniyor...")
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

import langchain_core.documents.base

def _safe_setstate(self, state):
    if "__fields_set__" in state:
        del state["__fields_set__"]
    self.__dict__.update(state)

langchain_core.documents.base.Document.__setstate__ = _safe_setstate

try:
    vector_db = FAISS.load_local(folder_path=FAISS_PATH, embeddings=embeddings, allow_dangerous_deserialization=True)
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})
    print(f"FAISS Veritabanı Başarıyla Yüklendi! Toplam Chunk: {vector_db.index.ntotal}")
except Exception as e:
    print(f"FAISS Yüklenirken Hata Oluştu. Dosya yolunu ({FAISS_PATH}) kontrol edin!\nHata detayı: {e}")
    # Hata verirse kodun devam edip çökmemesi için boş bir retriever koy (önlem olarak)
    retriever = None
    
# =====================================================================
# HER KATEGORİDEN 2 RASTGELE ÖRNEK SEÇİP KAYDETME
# =====================================================================
print("\nÖrnek sorular seçiliyor...")
sample_list = []
kategoriler = df_qa['question_type'].dropna().unique()

for kat in kategoriler:
    df_kat = df_qa[df_qa['question_type'] == kat]
    ornekler = df_kat.sample(n=min(2, len(df_kat)), random_state=42)
    sample_list.append(ornekler)

df_ornekler = pd.concat(sample_list)
df_ornekler_export = df_ornekler.rename(columns={'instruction': 'Question', 'output': 'Gold Answer', 'question_type': 'Q_Type', 'doc_id': 'Doc_No'})
df_ornekler_export.to_excel("Ornek_Sorular_Listesi.xlsx", index=False)
print("Her kategoriden (OOD/Halüsinasyon dahil) 2'şer adet örnek seçilip 'Ornek_Sorular_Listesi.xlsx' olarak kaydedildi!\n")


# =====================================================================
# 2. METRİK VE HAKEM (JUDGE) FONKSİYONLARI
# =====================================================================
def calc_em_f1(prediction, ground_truth):
    pred_norm = str(prediction).lower().translate(str.maketrans('', '', string.punctuation)).strip()
    truth_norm = str(ground_truth).lower().translate(str.maketrans('', '', string.punctuation)).strip()
    em = 1.0 if (truth_norm and truth_norm in pred_norm) else 0.0
    
    pred_tokens, truth_tokens = pred_norm.split(), truth_norm.split()
    if not pred_tokens or not truth_tokens: return em, 0.0
    
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0: return em, 0.0
    
    precision, recall = num_same / len(pred_tokens), num_same / len(truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return em, f1


judge_prompt = PromptTemplate(
    template="""Context: {context}\nGold Answer: {gold}\nModel Answer: {answer}\n
Evaluate and output ONLY 4 space-separated numbers for Relevance(1-5), Faithfulness(1-5), Fluency(1-5), and Hallucination(0-1). Example: 4 5 5 0""",
    input_variables=["context", "gold", "answer"]
)
judge_chain = judge_prompt | llm_judge

def get_human_metrics(context, gold, answer):
    try:
        res = judge_chain.invoke({"context": context if context else "NO RAG", "gold": gold, "answer": answer}).content.strip()
        scores = [int(s) for s in res.split() if s.isdigit()]
        if len(scores) >= 4: return scores[0], scores[1], scores[2], scores[3]
    except: pass
    return 0, 0, 0, 0

# =====================================================================
# 3. TEST DÖNGÜSÜ (5 SİSTEM)
# =====================================================================
results = []
test_data = df_qa.to_dict('records')

# Sistem Listesi 
systems = ["Retriever-Only", "LLM-Only (Base)", "Fine-Tuned LLM", "RAG + Base LLM", "RAG + Fine-Tuned LLM"]
gpu_hourly_cost = 0.30 # Örnek T4 GPU saatlik maliyeti (Dolar)


try:
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})
except NameError:
    print("UYARI: 'vector_db' bulunamadı. Önceki FAISS veritabanı hücresini çalıştırdığınızdan emin olun!")

for row in tqdm(test_data):
    q = row['instruction']
    gold = row['output']
    q_type = row.get('question_type', 'NORMAL')
    doc_id = row.get('doc_id', 'Bilinmiyor') 
    
    for sys_name in systems:
        torch.cuda.empty_cache()
        start_vram = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0.0
        start_ram = psutil.virtual_memory().used / (1024**3)
        t0 = time.perf_counter()
        
        # A) RETRIEVER-ONLY SENARYOSU
        if sys_name == "Retriever-Only":
            docs = retriever.invoke(q) 
            latency = time.perf_counter() - t0
            vram_used = (torch.cuda.memory_allocated() / (1024**3)) - start_vram
            ram_used = (psutil.virtual_memory().used / (1024**3)) - start_ram
            cost = (latency / 3600) * gpu_hourly_cost
            
            results.append({
                "Question_Type": q_type,
                "Doc_ID": doc_id,
                "Soru": q,
                "Gold_Cevap": gold,
                "Model_Cevabi": "-", 
                "System": sys_name, "EM": "-", "F1": "-", "Relevance": "-", "Faithfulness": "-", 
                "Fluency": "-", "Hallucination": "-", "Latency(s)": latency, "Cost($)": cost, 
                "VRAM(GB)": max(vram_used, 0.01), "SysRAM(GB)": max(ram_used, 0.1), "Token/s": "-"
            })
            continue

        # B) LLM SENARYOLARI
        is_rag = "RAG" in sys_name
        is_ft = "Fine-Tuned" in sys_name
        
        # RAG ise context çek
        context = ""
        if is_rag:
            docs = retriever.invoke(q)
            context_texts = []
            for d in docs:
                if hasattr(d, 'page_content'):
                    context_texts.append(d.page_content)
                elif hasattr(d, '__dict__') and '__dict__' in d.__dict__:
                    context_texts.append(d.__dict__['__dict__'].get('page_content', ''))
                else:
                    context_texts.append(str(d))
            context = "\n".join(context_texts)
        
        model = ft_model if is_ft else base_model
        prompt = f"Context: {context}\n\nQuestion: {q}" if is_rag else q
        
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150)
        
        latency = time.perf_counter() - t0
        vram_used = (torch.cuda.memory_allocated() / (1024**3)) - start_vram
        ram_used = (psutil.virtual_memory().used / (1024**3)) - start_ram
        gen_tokens = len(outputs[0]) - len(inputs[0])
        token_s = gen_tokens / latency if latency > 0 else 0
        cost = (latency / 3600) * gpu_hourly_cost
        
        response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True).strip() # Promptu çıkarıp sadece cevabı aldık
        em, f1 = calc_em_f1(response, gold)
        rel, faith, flu, hall = get_human_metrics(context, gold, response)
        
        results.append({
            "Question_Type": q_type,
            "Doc_ID": doc_id,
            "Soru": q,
            "Gold_Cevap": gold,
            "Model_Cevabi": response,
            "System": sys_name, "EM": em, "F1": f1, "Relevance": rel, "Faithfulness": faith, 
            "Fluency": flu, "Hallucination": hall, "Latency(s)": latency, "Cost($)": cost, 
            "VRAM(GB)": max(vram_used, 0.1), "SysRAM(GB)": max(ram_used, 0.1), "Token/s": token_s
        })

# =====================================================================
# 4. TABLOYU FORMATLAMA VE EXCEL'E KAYDETME
# =====================================================================
df_all = pd.DataFrame(results)


df_all.to_excel("Tum_Sorularin_Detayli_Cevaplari.xlsx", index=False)

# Retriever-Only hariç tutarak sayısal ortalamaları alma
df_llms = df_all[df_all["System"] != "Retriever-Only"].copy()
for col in ["EM", "F1", "Relevance", "Faithfulness", "Fluency", "Hallucination", "Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)", "Token/s"]:
    df_llms[col] = pd.to_numeric(df_llms[col], errors='coerce')

summary_llms = df_llms.groupby("System").mean().round(3).reset_index()

# Retriever-Only için özel hesaplama (Sadece donanım ve hız, diğerleri "-")
df_retriever = df_all[df_all["System"] == "Retriever-Only"].copy()
for col in ["Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)"]:
    df_retriever[col] = pd.to_numeric(df_retriever[col], errors='coerce')

ret_summary = pd.DataFrame([{
    "System": "Retriever-Only", "EM": "-", "F1": "-", "Relevance": "-", "Faithfulness": "-", 
    "Fluency": "-", "Hallucination": "-", 
    "Latency(s)": round(df_retriever["Latency(s)"].mean(), 3),
    "Cost($)": round(df_retriever["Cost($)"].mean(), 5),
    "VRAM(GB)": round(df_retriever["VRAM(GB)"].mean(), 3),
    "SysRAM(GB)": round(df_retriever["SysRAM(GB)"].mean(), 3),
    "Token/s": "-"
}])

# İkisini birleştir
final_table = pd.concat([ret_summary, summary_llms], ignore_index=True)

system_order = ["Retriever-Only", "LLM-Only (Base)", "Fine-Tuned LLM", "RAG + Base LLM", "RAG + Fine-Tuned LLM"]
final_table['System'] = pd.Categorical(final_table['System'], categories=system_order, ordered=True)
final_table = final_table.sort_values('System')

print("\n" + "="*95)
print("NİHAİ KARŞILAŞTIRMA TABLOSU")
print("="*95)
print(final_table.to_string(index=False))
print("="*95)

# Excel Olarak Kaydet
excel_filename = "10_Madde_Nihai_Sistem_Karsilastirmasi.xlsx"
final_table.to_excel(excel_filename, index=False, engine='openpyxl')
print(f"Tablo eksiksiz olarak '{excel_filename}' dosyasına kaydedildi.")

In [ ]:
import pandas as pd

print("Kaydedilmiş sonuçlar okunuyor ve tablo düzeltiliyor...")


df_all = pd.read_excel("Tum_Sorularin_Detayli_Cevaplari.xlsx")

# Retriever-Only hariç tutarak SAYISAL ortalamaları alma
df_llms = df_all[df_all["System"] != "Retriever-Only"].copy()
for col in ["EM", "F1", "Relevance", "Faithfulness", "Fluency", "Hallucination", "Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)", "Token/s"]:
    df_llms[col] = pd.to_numeric(df_llms[col], errors='coerce')


summary_llms = df_llms.groupby("System").mean(numeric_only=True).round(3).reset_index()

# Retriever-Only için özel hesaplama 
df_retriever = df_all[df_all["System"] == "Retriever-Only"].copy()
for col in ["Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)"]:
    df_retriever[col] = pd.to_numeric(df_retriever[col], errors='coerce')

ret_summary = pd.DataFrame([{
    "System": "Retriever-Only", "EM": "-", "F1": "-", "Relevance": "-", "Faithfulness": "-", 
    "Fluency": "-", "Hallucination": "-", 
    "Latency(s)": round(df_retriever["Latency(s)"].mean(), 3),
    "Cost($)": round(df_retriever["Cost($)"].mean(), 5),
    "VRAM(GB)": round(df_retriever["VRAM(GB)"].mean(), 3),
    "SysRAM(GB)": round(df_retriever["SysRAM(GB)"].mean(), 3),
    "Token/s": "-"
}])

# İkisini birleştir
final_table = pd.concat([ret_summary, summary_llms], ignore_index=True)

system_order = ["Retriever-Only", "LLM-Only (Base)", "Fine-Tuned LLM", "RAG + Base LLM", "RAG + Fine-Tuned LLM"]
final_table['System'] = pd.Categorical(final_table['System'], categories=system_order, ordered=True)
final_table = final_table.sort_values('System')

print("\n" + "="*95)
print("10. MADDE: NİHAİ KARŞILAŞTIRMA TABLOSU")
print("="*95)
print(final_table.to_string(index=False))
print("="*95)

# Excel Olarak Kaydet
excel_filename = "10_Madde_Nihai_Sistem_Karsilastirmasi.xlsx"
final_table.to_excel(excel_filename, index=False, engine='openpyxl')
print(f"\nTablo eksiksiz olarak '{excel_filename}' dosyasına kaydedildi.")

In [4]:
import pandas as pd
import re
import time
from tqdm import tqdm
from langchain_cohere import ChatCohere
from langchain_core.prompts import PromptTemplate
from kaggle_secrets import UserSecretsClient

print("Hakem (Cohere) Puanları Hız Sınırı Korumasıyla Yeniden Hesaplanıyor...")


user_secrets = UserSecretsClient()
cohere_api_key = user_secrets.get_secret("COHERE_API_KEY")
llm_judge = ChatCohere(model="command-r-plus-08-2024", temperature=0.0, cohere_api_key=cohere_api_key)

df = pd.read_excel("/kaggle/input/datasets/sudeozmen/cevaplar/Tum_Sorularin_Detayli_Cevaplari.xlsx")

judge_prompt = PromptTemplate(
    template="Gold Answer: {gold}\nModel Answer: {answer}\nEvaluate the model answer. Output ONLY 4 integers separated by spaces for Relevance(1-5), Faithfulness(1-5), Fluency(1-5), and Hallucination(0-1). Example: 4 5 5 0",
    input_variables=["gold", "answer"]
)
judge_chain = judge_prompt | llm_judge

def extract_scores(res_text):
    nums = re.findall(r'\d+', res_text)
    if len(nums) >= 4:
        return int(nums[0]), int(nums[1]), int(nums[2]), int(nums[3])
    return None

# Toplam değerlendirilecek satır
df_to_score = df[df['System'] != 'Retriever-Only'].copy()


for idx, row in tqdm(df_to_score.iterrows(), total=len(df_to_score)):
    # Eğer bu satır daha önce başarıyla puanlandıysa atla
    if pd.notna(df.at[idx, 'Relevance']) and df.at[idx, 'Relevance'] > 0:
        continue
        
    try:
        res = judge_chain.invoke({"gold": str(row['Gold_Cevap']), "answer": str(row['Model_Cevabi'])}).content
        scores = extract_scores(res)
        
        if scores:
            df.at[idx, 'Relevance'], df.at[idx, 'Faithfulness'], df.at[idx, 'Fluency'], df.at[idx, 'Hallucination'] = scores
        
        time.sleep(4) 
        
    except Exception as e:
        print(f"\n[Satır {idx}] API Hatası: {e}")
        print("API soğuması için 15 saniye bekleniyor...")
        time.sleep(15) # Hata alırsa daha uzun bekle

# Dosyayı güncel haliyle kaydet
df.to_excel("Tum_Sorularin_Detayli_Cevaplari.xlsx", index=False)

# ---- TABLOYU YENİDEN ÇİZ ----
df_llms = df[df["System"] != "Retriever-Only"].copy()
for col in ["EM", "F1", "Relevance", "Faithfulness", "Fluency", "Hallucination", "Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)", "Token/s"]:
    df_llms[col] = pd.to_numeric(df_llms[col], errors='coerce')

summary_llms = df_llms.groupby("System").mean(numeric_only=True).round(3).reset_index()

df_retriever = df[df["System"] == "Retriever-Only"].copy()
for col in ["Latency(s)", "Cost($)", "VRAM(GB)", "SysRAM(GB)"]:
    df_retriever[col] = pd.to_numeric(df_retriever[col], errors='coerce')

ret_summary = pd.DataFrame([{
    "System": "Retriever-Only", "EM": "-", "F1": "-", "Relevance": "-", "Faithfulness": "-", 
    "Fluency": "-", "Hallucination": "-", 
    "Latency(s)": round(df_retriever["Latency(s)"].mean(), 3),
    "Cost($)": round(df_retriever["Cost($)"].mean(), 5),
    "VRAM(GB)": round(df_retriever["VRAM(GB)"].mean(), 3),
    "SysRAM(GB)": round(df_retriever["SysRAM(GB)"].mean(), 3),
    "Token/s": "-"
}])

final_table = pd.concat([ret_summary, summary_llms], ignore_index=True)
system_order = ["Retriever-Only", "LLM-Only (Base)", "Fine-Tuned LLM", "RAG + Base LLM", "RAG + Fine-Tuned LLM"]
final_table['System'] = pd.Categorical(final_table['System'], categories=system_order, ordered=True)
final_table = final_table.sort_values('System')

print("\n" + "="*95)
print("10. MADDE: NİHAİ BAŞARI TABLOSU (İNSAN METRİKLERİ DAHİL)")
print("="*95)
print(final_table.to_string(index=False))

Hakem (Cohere) Puanları Hız Sınırı Korumasıyla Yeniden Hesaplanıyor...


100%|██████████| 600/600 [46:58<00:00,  4.70s/it] 



10. MADDE: NİHAİ BAŞARI TABLOSU (İNSAN METRİKLERİ DAHİL)
              System     EM     F1 Relevance Faithfulness Fluency Hallucination  Latency(s)  Cost($)  VRAM(GB)  SysRAM(GB) Token/s
      Retriever-Only      -      -         -            -       -             -       0.016    0.000      0.01         0.1       -
     LLM-Only (Base)  0.027  0.129     3.353        3.207   4.287         0.007      16.933    0.001      0.10         0.1   7.033
      Fine-Tuned LLM  0.027  0.129      3.28        3.107   4.227         0.007      16.931    0.001      0.10         0.1   7.033
      RAG + Base LLM    0.2   0.17     4.333         4.38   4.733         0.013      24.988    0.002      0.10         0.1   5.883
RAG + Fine-Tuned LLM    0.2   0.17      4.32        4.327    4.72         0.013      24.905    0.002      0.10         0.1   5.902
